# 01 — Primeira previsão

Neste notebook você vai: carregar uma série de tráfego, olhar os dados, pedir uma previsão ao TimesFM 3.0 e desenhar o resultado.

**Como usar:** clique numa célula e aperte `Shift + Enter` para executá-la, uma de cada vez, de cima para baixo. A primeira previsão baixa o modelo e demora mais.

## 1. Carregar os dados

Usamos o arquivo sintético que acompanha o projeto: contagem de veículos por hora durante 8 semanas.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from trafego_lab.dados import detect_columns, prepare_series, resample_series

csv_path = Path('..') / 'data' / 'samples' / 'trafego_sintetico.csv'
df = pd.read_csv(csv_path)
df.head()

## 2. Preparar a série

`detect_columns` adivinha qual coluna é a data e qual é o volume. `prepare_series` ordena e limpa; `resample_series` garante um ponto por hora, sem buracos.

In [ ]:
date_col, value_col = detect_columns(df)
print('Data:', date_col, '| Volume:', value_col)

series = resample_series(prepare_series(df, date_col, value_col), 'h', how='sum')
series.plot(figsize=(12, 3), title='Contagem horária de veículos')
plt.show()

Repare nos picos de manhã e no fim da tarde e no padrão mais baixo no fim de semana. É isso que o modelo vai tentar reconhecer.

## 3. Carregar o modelo

`load_forecaster` escolhe sozinho o motor: MLX no Mac com Apple silicon.

In [ ]:
from trafego_lab.modelo import choose_backend, load_forecaster

print('Motor:', choose_backend())
model = load_forecaster()

## 4. Prever as próximas 48 horas

In [ ]:
from trafego_lab.modelo import forecast

horizon = 48
result = forecast(model, series.to_numpy(), horizon)
print('Previsão pontual:', result.point.shape, '| quantis:', result.quantiles.shape)

A previsão pontual tem 48 valores (um por hora). Os quantis têm 9 colunas: 10%, 20%, … 90%.

## 5. Desenhar

In [ ]:
future = pd.date_range(series.index[-1], periods=horizon + 1, freq='h')[1:]

fig, ax = plt.subplots(figsize=(12, 4))
series.iloc[-24 * 7:].plot(ax=ax, label='Histórico', color='gray')
ax.plot(future, result.point, label='Previsão TimesFM', color='tab:blue', lw=2)
ax.fill_between(future, result.lower, result.upper, alpha=0.2, label='Faixa 10%-90%')
ax.legend()
plt.show()

## Para experimentar

- Troque `horizon` por 24 e por 168. O que acontece com a faixa azul?
- Use só `series.iloc[-72:]` como histórico. A previsão piora?
- Depois de baixar um dataset do Kaggle, troque o `csv_path`.